In [1]:
# Create spark session

from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Streaming from Kafka")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0')
    .config("spark.sql.shuffle.partitions", 8)
    .master("local[2]")
    .getOrCreate()
)

spark

In [9]:
# Create the kafka_df to read from kafka

kafka_df = (
    spark
    .read
    .format("kafka")
    .option("kafka.bootstrap.servers", "ed-kafka:29092")
    .option("subscribe", "device-data")
    .option("startingOffsets", "earliest")
    .load()
)
kafka_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [10]:
kafka_df.show()

+----+--------------------+-----------+---------+------+--------------------+-------------+
| key|               value|      topic|partition|offset|           timestamp|timestampType|
+----+--------------------+-----------+---------+------+--------------------+-------------+
|null|[7B 22 65 76 65 6...|device-data|        0|     0|2026-04-09 13:24:...|            0|
+----+--------------------+-----------+---------+------+--------------------+-------------+



In [11]:
# Parse value from binary to string into kafka_json_df
from pyspark.sql.functions import expr

kafka_json_df = kafka_df.withColumn("value", expr("cast(value as string)"))

In [12]:
kafka_json_df.show()

+----+--------------------+-----------+---------+------+--------------------+-------------+
| key|               value|      topic|partition|offset|           timestamp|timestampType|
+----+--------------------+-----------+---------+------+--------------------+-------------+
|null|{"eventId": "e3cb...|device-data|        0|     0|2026-04-09 13:24:...|            0|
+----+--------------------+-----------+---------+------+--------------------+-------------+



In [13]:
# Schema for payload

from struct import Struct
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, LongType

json_schema = (
    StructType(
        [StructField("customerId", StringType(), True),
        StructField("data",StructType(
            [StructField("devices",
                        # Array of device objects
                        ArrayType(StructType([
                            StructField("deviceId", StringType(), True),
                            StructField("measure", StringType(), True),
                            StructField("status", StringType(), True),
                            StructField("temperature", LongType(), True),
                        ]), True), True)
                ]), True),
        StructField("eventID", StringType(), True),
        StructField("eventOffset", LongType(), True),
        StructField("eventPublisher", StringType(), True),
        StructField("eventTime", StringType(), True)
        ])
)

In [14]:
# Apply the schema to payload to read the data
from pyspark.sql.functions import from_json, col

streaming_df = kafka_json_df.withColumn("values_json", from_json(col("value"), json_schema)).selectExpr("values_json.*")

In [15]:
streaming_df.printSchema()
streaming_df.show()

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventID: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)

+----------+--------------------+-------+-----------+--------------+--------------------+
|customerId|                data|eventID|eventOffset|eventPublisher|           eventTime|
+----------+--------------------+-------+-----------+--------------+--------------------+
|   CI00103|{[{D001, C, ERROR...|   null|      10001|        device|2023-01-05 11:13:...|
+----------+--------------------+-------+-----------+--------------+-------

In [16]:
# lets explode the data as devices contains list/array of device reading

from pyspark.sql.functions import explode

exploded_df = streaming_df.withColumn("data_devices", explode("data.devices")).drop("data")

In [17]:
# Flatten the exploded df
from pyspark.sql.functions import col
flattened_df = (
    exploded_df
    .withColumn("deviceID", col("data_devices.deviceId"))
    .withColumn("measure", col("data_devices.measure"))
    .withColumn("status", col("data_devices.status"))
    .withColumn("temperature", col("data_devices.temperature"))
    .drop("data_devices")
)

In [18]:
flattened_df.printSchema()
flattened_df.show()

root
 |-- customerId: string (nullable = true)
 |-- eventID: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- deviceID: string (nullable = true)
 |-- measure: string (nullable = true)
 |-- status: string (nullable = true)
 |-- temperature: long (nullable = true)

+----------+-------+-----------+--------------+--------------------+--------+-------+-------+-----------+
|customerId|eventID|eventOffset|eventPublisher|           eventTime|deviceID|measure| status|temperature|
+----------+-------+-----------+--------------+--------------------+--------+-------+-------+-----------+
|   CI00103|   null|      10001|        device|2023-01-05 11:13:...|    D001|      C|  ERROR|         15|
|   CI00103|   null|      10001|        device|2023-01-05 11:13:...|    D002|      C|SUCCESS|         16|
+----------+-------+-----------+--------------+--------------------+--------+-------+-------+----

In [ ]:
# Write the output to console sink to check the output

(flattened_df
 .write
 .format("console")
 .outputMode("append")
 .option("checkpointLocation", "checkpoint_dir_kafka")
 .start()
 .awaitTermination())